# Nik Studio - video model test

**Two cells. Run cell 1, then cell 2. That is the whole thing.**

It takes one picture of your character and makes a short clip where he
actually moves. Nothing here touches your project.

This is a **test, not the tool**. It answers one question before any code
gets written: can a free Colab GPU animate your character well enough to
be worth building on?

Before you start: **Runtime > Change runtime type > T4 GPU**.

Cell 2 takes about 12 minutes the first time, because a 9GB model has to
come down. Run it a second time - to try a different prompt - and it is
about 3 minutes, because the model stays loaded.


In [ ]:
# ======================================================================
# CELL 1 of 2 - the packages
# ======================================================================
#
# If Colab offers "RESTART SESSION" when this finishes, click it.
# Cell 2 does not depend on anything in here, so a restart costs nothing.

!pip install -q "diffusers>=0.32" "transformers>=4.44" accelerate safetensors sentencepiece bitsandbytes imageio-ffmpeg

print("Packages installed. Now run cell 2.")


In [ ]:
# ======================================================================
# CELL 2 of 2 - the whole test
# ======================================================================
#
# Picks up your picture, loads the model, makes the clip, plays it back.
#
# Nothing above this line matters to it, so it cannot be broken by
# running the cells out of order or by Colab restarting the runtime.
#
# Run it again to try a different prompt: the picture and the model are
# both kept, so the second run takes about three minutes instead of
# twelve. Change PROMPT below and press play.

import gc
import time

from pathlib import Path

import torch

from PIL import Image


# ---------------------------------------------------------------- SETTINGS

# What should HAPPEN. Not what the picture shows - that is already there.
# One clear physical action beats three vague ones.
PROMPT = (
    "The little boy claps his hands together and bounces up and down "
    "with excitement, laughing. His head tilts and his arms swing. Soap "
    "bubbles drift slowly upward around him. The camera stays still. "
    "Pixar style 3D animation, smooth natural motion, bright and cheerful."
)

NEGATIVE = (
    "worst quality, blurry, jittery, distorted, deformed face, "
    "static image, no movement, extra limbs, watermark, text"
)

# LTX needs both sides to be a multiple of 32. 704x384 is close to 16:9
# and deliberately small - this run is to find out whether the child
# moves at all. Bigger comes after that works.
WIDTH, HEIGHT = 704, 384

# 65 frames at 24fps is under three seconds. Short on purpose: decoding
# is where the memory goes, and a long clip is what kills the session.
FRAMES, FPS, STEPS = 65, 24, 40

MODEL = "Lightricks/LTX-Video"


# ------------------------------------------------------------------- GPU

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime > Change runtime type > T4 GPU, then run again."
    )

major, _ = torch.cuda.get_device_capability()

# Do not ask torch.cuda.is_bf16_supported() - it answers True on a T4,
# because torch emulates bfloat16 in software rather than refusing, and
# that emulation is slower than it is worth. Ask the hardware instead:
# compute capability 8.0 (Ampere) or newer is where bfloat16 is real.
DTYPE = torch.bfloat16 if major >= 8 else torch.float16

print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Precision:", "bfloat16" if major >= 8 else "float16 (T4)")


# --------------------------------------------------------------- PICTURE

if "IMAGE" not in globals():

    from google.colab import files

    print("\nUpload one image - Images\\Scene01.png from your episode:")

    picture = Image.open(
        Path("/content") / next(iter(files.upload()))
    ).convert("RGB")

    # Cover and crop rather than squash. A stretched child is not a fair
    # test of the model.
    scale = max(WIDTH / picture.width, HEIGHT / picture.height)

    picture = picture.resize(
        (round(picture.width * scale), round(picture.height * scale)),
        Image.LANCZOS,
    )

    left = (picture.width - WIDTH) // 2
    top = (picture.height - HEIGHT) // 2

    IMAGE = picture.crop((left, top, left + WIDTH, top + HEIGHT))

display(IMAGE)


# ----------------------------------------------------------------- MODEL

# Kept between runs. Loading it is ten of the twelve minutes, and there
# is no reason to pay that again just to change a prompt.
if "pipe" not in globals():

    from diffusers import LTXImageToVideoPipeline

    from transformers import BitsAndBytesConfig, T5EncoderModel

    print("\nLoading the model. About ten minutes the first time.")

    started = time.time()

    # The 9GB text encoder is what crashed the earlier attempt at this:
    # a free Colab has 12.7GB of ordinary RAM, and that is not enough to
    # hold it alongside everything else. Loading it in 8-bit sends it
    # straight to the GPU at about 4.7GB, never through RAM at full size.
    text_encoder = T5EncoderModel.from_pretrained(
        MODEL,
        subfolder="text_encoder",
        quantization_config=BitsAndBytesConfig(load_in_8bit=True),
        device_map="auto",
    )

    pipe = LTXImageToVideoPipeline.from_pretrained(
        MODEL,
        text_encoder=text_encoder,
        torch_dtype=DTYPE,
    )

    # The text encoder is already on the card in 8-bit; move the rest.
    pipe.transformer.to("cuda")
    pipe.vae.to("cuda")

    # Decoding every frame in one piece is what runs the card out of
    # memory. Tiling decodes it in patches instead.
    pipe.vae.enable_tiling()

    print(f"Model ready in {time.time() - started:.0f}s "
          f"({torch.cuda.memory_allocated() / 1e9:.1f}GB on the card).")

else:
    print("\nModel already loaded - reusing it.")


# ------------------------------------------------------------ GENERATION

def make(count):

    return pipe(
        image=IMAGE,
        prompt=PROMPT,
        negative_prompt=NEGATIVE,
        width=WIDTH,
        height=HEIGHT,
        num_frames=count,
        frame_rate=FPS,
        num_inference_steps=STEPS,
        guidance_scale=3.0,
        generator=torch.Generator("cpu").manual_seed(42),
    ).frames[0]


print("\nGenerating. Two or three minutes.")

started = time.time()

# The model leaves only a couple of GB free on a T4, which is enough for
# a short clip and not always enough for this one. Come back with a
# shorter clip rather than die - half a result beats a dead session.
try:
    video = make(FRAMES)

except torch.cuda.OutOfMemoryError:

    gc.collect()
    torch.cuda.empty_cache()

    print("Card ran out of room - trying a shorter clip instead.")

    video = make(33)

print(f"Done in {(time.time() - started) / 60:.1f} minutes.")


# ------------------------------------------------------------------ WATCH

from diffusers.utils import export_to_video

from IPython.display import Video

clip = Path("/content/Scene01_test.mp4")

export_to_video(video, str(clip), fps=FPS)

print("Saved:", clip, "- download it from the folder icon on the left.")

display(Video(str(clip), embed=True, width=704))


# ----------------------------------------------------------------------
# One question: does the child MOVE, and is he still your character?
#
#   Yes                  -> the video backend gets built.
#   Face changed/melted  -> fixable; fewer steps or a shorter clip.
#   Barely moves         -> the prompt needs a stronger physical action.
#   Smeary or flickering -> float16 on a T4, or the clip is too long.
# ----------------------------------------------------------------------
